# Copper certificate valuation
## Primary result: certificate vs physical

The objective is to measure the certificate premium or discount to the domestic physical benchmark.
The main chart uses the approved bounded, interpolated physical-to-intrinsic ratio method:
**100 × (certificate price / estimated physical price − 1)**.
Dots identify observed physical anchors; the remaining eligible dates use the approved interpolation.
Positive values indicate a premium; negative values indicate a discount.

Sources: IME certificate and physical transactions, Westmetall LME quotations, and the shared free-market USD/IRR series.
This presentation reads existing processed outputs; refresh the project before reviewing it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

WORKSPACE = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'shared').is_dir() and (p / 'commodity/copper').is_dir())
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
from shared.notebook_tools.commodity_dashboard import valuation_figures, PLOTLY_CONFIG
PROJECT = WORKSPACE / 'commodity/copper'
approved_primary = pd.read_csv(PROJECT / 'data/processed/bubble/copper_certificate_bubble.csv', parse_dates=['date'])
figures = valuation_figures(PROJECT, 'Copper')
figures[0].show(config=PLOTLY_CONFIG)
values = approved_primary['certificate_bubble_pct']
display(pd.DataFrame([{'Observations': len(values), 'From': approved_primary['date'].min(),
    'To': approved_primary['date'].max(), 'Mean (%)': values.mean(),
    'Median (%)': values.median(), 'Latest (%)': approved_primary.sort_values('date')['certificate_bubble_pct'].iloc[-1]}]))


## Supporting result: certificate vs intrinsic

Intrinsic reference = LME cash USD/kg × USD/IRR. This is a separate comparison, not the primary certificate-to-physical bubble.

In [ ]:
figures[1].show(config=PLOTLY_CONFIG)

## Supporting result: physical vs intrinsic

This chart measures the domestic physical-market premium or discount to the same international reference.

In [ ]:
figures[2].show(config=PLOTLY_CONFIG)

## Reading the results

Use the first chart for the main certificate-to-physical comparison. The next two charts provide international-value context and are not interchangeable with it. Statistics above are calculated from the loaded data rather than copied into prose.

---
# Research appendix — supporting work and experimental methods

The following work records how the benchmark and alternative methods were investigated. Experimental regressions do not replace the approved primary result above.

## Historical regression investigation

Research record only: the approved result is presented above. The later intrinsic-only regression supersedes the early two-feature experiment within this appendix.

In [ ]:
from bisect import bisect_right
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. Load the four source datasets

In [ ]:
def first_existing(*candidates):
    path = next((Path(candidate) for candidate in candidates if Path(candidate).exists()), None)
    if path is None:
        raise FileNotFoundError(f"None of these paths exists: {candidates}")
    return path


certificate_path = first_existing(
    "../data/raw/certificate/copper_certificate_raw.csv",
    "commodity/copper/data/raw/certificate/copper_certificate_raw.csv",
)
physical_path = first_existing(
    "../data/processed/physical/nci_copper_cash_daily.csv",
    "commodity/copper/data/processed/physical/nci_copper_cash_daily.csv",
)
lme_path = first_existing(
    "../data/raw/lme/copper_lme_raw.csv",
    "commodity/copper/data/raw/lme/copper_lme_raw.csv",
)
usd_path = first_existing(
    "../../../shared/data/raw/fx/usd_to_rial.csv",
    "shared/data/raw/fx/usd_to_rial.csv",
)

certificate_raw = pd.read_csv(certificate_path)
physical_daily = pd.read_csv(physical_path)
lme_raw = pd.read_csv(lme_path)
usd_raw = pd.read_csv(usd_path)

{
    "certificate_raw": certificate_raw.shape,
    "physical_daily": physical_daily.shape,
    "lme_raw": lme_raw.shape,
    "usd_raw": usd_raw.shape,
}

## 2. Clean dates and prices

Certificate price is daily VWAP (`TradesValue / TradesVolume`). LME is converted from USD/metric-tonne to USD/kg. Missing weekend or holiday LME/USD observations are joined backward, never from a future date.

In [ ]:
certificate = certificate_raw.copy()
certificate["date"] = pd.to_datetime(certificate["DT"].str[:10])
for column in ["TradesVolume", "TradesValue", "TodaySettlementPrice"]:
    certificate[column] = pd.to_numeric(certificate[column], errors="raise")
certificate = certificate.loc[certificate["TradesVolume"] > 0].copy()
certificate["certificate_price"] = (
    certificate["TradesValue"] / certificate["TradesVolume"]
)
certificate["settlement_check_error"] = (
    certificate["certificate_price"] - certificate["TodaySettlementPrice"]
).abs()
assert certificate["settlement_check_error"].max() <= 0.500001
certificate = certificate[[
    "date", "certificate_price", "TradesVolume", "TradesValue"
]].sort_values("date")

physical = physical_daily.copy()
physical["date"] = pd.to_datetime(physical["physical_trade_date_gregorian"])
physical["physical_price"] = pd.to_numeric(
    physical["physical_weighted_price"], errors="raise"
)
physical = physical[["date", "physical_price", "total_quantity", "physical_trades_value_irr"]].sort_values("date")

lme = lme_raw.loc[lme_raw["cash_settlement"].astype(str).str.strip().ne("-")].copy()
lme["lme_source_date"] = pd.to_datetime(lme["date"])
lme["lme_usd_per_ton"] = pd.to_numeric(
    lme["cash_settlement"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
lme["lme_usd_per_kg"] = lme["lme_usd_per_ton"] / 1_000
lme = lme[["lme_source_date", "lme_usd_per_kg"]].sort_values("lme_source_date")

def parse_mixed_gregorian(value):
    parts = [int(part) for part in str(value).replace("-", "/").split("/")]
    if parts[0] >= 1900:
        year, month, day = parts
    else:
        month, day, year = parts
    return pd.Timestamp(year=year, month=month, day=day)

usd = usd_raw.copy()
usd["usd_source_date"] = usd["date_gr"].map(parse_mixed_gregorian)
usd["usd_irr"] = pd.to_numeric(
    usd["price_irr"].astype(str).str.replace(",", "", regex=False),
    errors="raise",
)
usd = usd[["usd_source_date", "usd_irr"]].sort_values("usd_source_date")

assert certificate["date"].is_unique
assert physical["date"].is_unique
assert lme["lme_source_date"].is_unique
assert usd["usd_source_date"].is_unique

## 3. Add backward-looking LME and USD inputs

In [ ]:
def add_market_inputs(frame):
    result = frame.sort_values("date").copy()
    result = pd.merge_asof(
        result,
        lme,
        left_on="date",
        right_on="lme_source_date",
        direction="backward",
    )
    result = pd.merge_asof(
        result.sort_values("date"),
        usd,
        left_on="date",
        right_on="usd_source_date",
        direction="backward",
    )
    result["lme_age_days"] = (result["date"] - result["lme_source_date"]).dt.days
    result["usd_age_days"] = (result["date"] - result["usd_source_date"]).dt.days
    result["intrinsic_price"] = result["lme_usd_per_kg"] * result["usd_irr"]
    if result[["lme_usd_per_kg", "usd_irr"]].isna().any().any():
        raise ValueError("Missing LME or USD inputs after backward merge")
    return result


# Regression training data comes only from the physical market, USD, and LME.
# The certificate dataset is not merged into the regression sample.
REGRESSION_START_DATE = pd.Timestamp("2025-11-12")
physical_regression_sample = physical.loc[
    physical["date"] >= REGRESSION_START_DATE
].copy()
anchors = add_market_inputs(physical_regression_sample)
assert not anchors.empty, "No physical observations found in the regression period"

# Certificate market inputs are prepared separately and are used only after fitting.
certificate_features = add_market_inputs(certificate)

anchors[[
    "date", "physical_price", "usd_irr", "lme_usd_per_kg", "intrinsic_price"
]]

## 4. Compare linear and degree-2 polynomial regression

The regression uses the currently available physical observations from the stated start date. The polynomial model is regularized with Ridge. `TimeSeriesSplit` preserves temporal order, and selection uses the lowest out-of-sample RMSE. Certificate prices are not present in `X`, `y`, or the regression sample.

In [ ]:
FEATURES = ["usd_irr", "lme_usd_per_kg"]
TARGET = "physical_price"

X = anchors[FEATURES]
y = anchors[TARGET]

models = {
    "linear": Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

tscv = TimeSeriesSplit(n_splits=5)
comparison_rows = []
oos_predictions = {}

for name, model in models.items():
    prediction = pd.Series(np.nan, index=y.index, dtype=float)
    for train_index, test_index in tscv.split(X):
        fold_model = clone(model)
        fold_model.fit(X.iloc[train_index], y.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X.iloc[test_index])
    valid = prediction.notna()
    valid_prediction = prediction.loc[valid]
    actual = y.loc[valid]
    oos_predictions[name] = valid_prediction
    comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, valid_prediction),
        "RMSE": mean_squared_error(actual, valid_prediction) ** 0.5,
        "R2": r2_score(actual, valid_prediction),
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
model_comparison

## 5. Fit the selected model and inspect anchor fit

In [ ]:
selected_model_name = model_comparison.loc[0, "model"]
selected_model = models[selected_model_name]
selected_model.fit(X, y)

anchors["fitted_physical_price"] = selected_model.predict(X)
anchors["fitted_error_pct"] = (
    anchors["fitted_physical_price"] / anchors["physical_price"] - 1
) * 100

print(f"Selected model: {selected_model_name}")
if selected_model_name == "polynomial_degree_2_ridge":
    print(f"Selected Ridge alpha: {selected_model.named_steps['model'].alpha_}")

anchors[[
    "date", "physical_price", "fitted_physical_price", "fitted_error_pct"
]]

In [ ]:
limits = [min(anchors['physical_price'].min(), anchors['fitted_physical_price'].min()),
          max(anchors['physical_price'].max(), anchors['fitted_physical_price'].max())]
fig = go.Figure()
fig.add_trace(go.Scatter(x=anchors['physical_price'], y=anchors['fitted_physical_price'],
    mode='markers', name='Physical anchors', marker=dict(color='#1976D2', size=8)))
fig.add_trace(go.Scatter(x=limits, y=limits, name='Perfect fit',
    line=dict(color='#455A64', dash='dash')))
fig.update_layout(title=f'Observed vs fitted IME physical price: {selected_model_name}',
    xaxis_title='Observed IRR/kg', yaxis_title='Fitted IRR/kg',
    height=550, template='plotly_white')
fig.show()

## 6. Estimate physical price for all available certificate trading days and calculate bubble

Unlike linear ratio interpolation, regression can produce estimates before the first and after the last physical anchor. These are model estimates—not observed physical trades—and should be interpreted with the cross-validation results.

In [ ]:
bubble_regression = certificate_features.copy()
bubble_regression["estimated_physical_price"] = selected_model.predict(
    bubble_regression[FEATURES]
)
if (bubble_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected regression produced a non-positive physical price")

bubble_regression["certificate_bubble_irr_per_kg"] = (
    bubble_regression["certificate_price"]
    - bubble_regression["estimated_physical_price"]
)
bubble_regression["certificate_bubble_pct"] = (
    bubble_regression["certificate_price"]
    / bubble_regression["estimated_physical_price"]
    - 1
) * 100
bubble_regression["is_physical_anchor_date"] = bubble_regression["date"].isin(anchors["date"])
bubble_regression["regression_model"] = selected_model_name

bubble_regression[[
    "date", "certificate_price", "estimated_physical_price",
    "certificate_bubble_pct", "is_physical_anchor_date"
]].describe(include="all")

In [ ]:
anchor_bubbles = bubble_regression.loc[bubble_regression['is_physical_anchor_date']]
fig = go.Figure()
fig.add_trace(go.Scatter(x=bubble_regression['date'], y=bubble_regression['certificate_bubble_pct'],
    name=f'Regression bubble ({selected_model_name})', line_color='#1976D2'))
fig.add_trace(go.Scatter(x=anchor_bubbles['date'], y=anchor_bubbles['certificate_bubble_pct'],
    mode='markers', name='Physical anchor date', marker=dict(color='#C62828', size=8)))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Daily copper certificate bubble: regression estimate',
    xaxis_title='Date', yaxis_title='Bubble (%)', height=550,
    template='plotly_white', hovermode='x unified')
fig.show()

## 7. Optional: inspect or save the experimental result

The save command is intentionally commented out. Uncomment it only if the regression result is accepted after reviewing cross-validation diagnostics.

In [ ]:
regression_output = bubble_regression[[
    "date",
    "certificate_price",
    "TradesVolume",
    "lme_source_date",
    "lme_age_days",
    "lme_usd_per_kg",
    "usd_source_date",
    "usd_age_days",
    "usd_irr",
    "intrinsic_price",
    "estimated_physical_price",
    "certificate_bubble_irr_per_kg",
    "certificate_bubble_pct",
    "is_physical_anchor_date",
    "regression_model",
]].copy()

regression_output.head()

# Optional save after reviewing model diagnostics:
# output_path = Path("data/processed/bubble/copper_certificate_bubble_regression.csv")
# output_path.parent.mkdir(parents=True, exist_ok=True)
# regression_output.to_csv(output_path, index=False, encoding="utf-8-sig")

## 8. Approved interpolated-ratio certificate bubble

This section plots the previously calculated production output in `copper_certificate_bubble.csv`. It is kept separate from the experimental regression bubble above.

In [ ]:
approved_bubble_path = first_existing(
    "../data/processed/bubble/copper_certificate_bubble.csv",
    "commodity/copper/data/processed/bubble/copper_certificate_bubble.csv",
)
approved_bubble = pd.read_csv(approved_bubble_path, parse_dates=["date"])
approved_bubble["certificate_bubble_pct"] = pd.to_numeric(
    approved_bubble["certificate_bubble_pct"], errors="raise"
)
approved_bubble = approved_bubble.sort_values("date").reset_index(drop=True)
approved_bubble["is_observed_ratio"] = approved_bubble["physical_ratio_method"].eq("observed")
print(
    f'Primary bubble: {len(approved_bubble):,} dates, '
    f'{approved_bubble["is_observed_ratio"].sum():,} exact anchors, '
    f'{(~approved_bubble["is_observed_ratio"]).sum():,} interpolated dates; '
    f'{approved_bubble["date"].min().date()} to {approved_bubble["date"].max().date()}'
)
print(
    f'Mean premium: {approved_bubble["certificate_bubble_pct"].mean():.2f}%; '
    f'median: {approved_bubble["certificate_bubble_pct"].median():.2f}%'
)

observed_ratio_days = approved_bubble.loc[approved_bubble['is_observed_ratio']]
fig = go.Figure()
fig.add_trace(go.Scatter(x=approved_bubble['date'], y=approved_bubble['certificate_bubble_pct'],
    name='Interpolated physical ratio', line_color='#1976D2'))
fig.add_trace(go.Scatter(x=observed_ratio_days['date'], y=observed_ratio_days['certificate_bubble_pct'],
    mode='markers', name='Observed physical ratio', marker=dict(color='#C62828', size=8)))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Copper certificate bubble: bounded physical-ratio estimate',
    xaxis_title='Date', yaxis_title='Bubble (%)', height=550,
    template='plotly_white', hovermode='x unified')
fig.show()

## 9. Observed IME physical-market premium over intrinsic copper value

For actual physical-market observations in the regression period, intrinsic value is `LME cash USD/kg × free-market USD/IRR`. The plotted premium is `(observed physical price / intrinsic price - 1) × 100`.

In [ ]:
physical_intrinsic_bubble = anchors[[
    "date", "physical_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
assert len(physical_intrinsic_bubble) == len(anchors)
physical_intrinsic_bubble["physical_vs_intrinsic_bubble_pct"] = (
    physical_intrinsic_bubble["physical_price"]
    / physical_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig = go.Figure(go.Scatter(x=physical_intrinsic_bubble['date'],
    y=physical_intrinsic_bubble['physical_vs_intrinsic_bubble_pct'], mode='lines+markers',
    name='Observed IME physical premium', line_color='#7B1FA2'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Observed IME physical copper vs intrinsic LME-FX value',
    xaxis_title='Physical trade date', yaxis_title='Premium / discount (%)',
    height=550, template='plotly_white')
fig.show()
physical_intrinsic_bubble

## 10. Certificate premium over intrinsic copper value

This measure compares certificate VWAP directly with `LME cash USD/kg × free-market USD/IRR`; it does not use observed, interpolated, or regression-estimated IME physical price.

In [ ]:
certificate_intrinsic_bubble = certificate_features[[
    "date", "certificate_price", "lme_usd_per_kg", "usd_irr", "intrinsic_price"
]].copy()
certificate_intrinsic_bubble["certificate_vs_intrinsic_bubble_pct"] = (
    certificate_intrinsic_bubble["certificate_price"]
    / certificate_intrinsic_bubble["intrinsic_price"]
    - 1
) * 100

fig = go.Figure(go.Scatter(x=certificate_intrinsic_bubble['date'],
    y=certificate_intrinsic_bubble['certificate_vs_intrinsic_bubble_pct'],
    name='Certificate premium over intrinsic value', line_color='#EF6C00'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Copper certificate vs intrinsic LME-FX value',
    xaxis_title='Date', yaxis_title='Premium / discount (%)',
    height=550, template='plotly_white')
fig.show()
certificate_intrinsic_bubble

## 11. Full-history IME physical copper premium over intrinsic value

This section extends the physical-market analysis to the complete available history. It uses every positive-trade observation in `nci_copper_cash_daily.csv`, not only the recent certificate period. Intrinsic value is `LME cash USD/kg × free-market USD/IRR`.

In [ ]:
physical_full_history = add_market_inputs(physical).copy()
physical_full_history["physical_vs_intrinsic_bubble_pct"] = (
    physical_full_history["physical_price"]
    / physical_full_history["intrinsic_price"]
    - 1
) * 100

assert len(physical_full_history) == len(physical)
assert physical_full_history["physical_vs_intrinsic_bubble_pct"].notna().all()

print(f"Physical observations: {len(physical_full_history):,}")
print(
    f"Coverage: {physical_full_history['date'].min().date()} "
    f"through {physical_full_history['date'].max().date()}"
)
physical_full_history[[
    "date",
    "physical_price",
    "lme_usd_per_kg",
    "usd_irr",
    "intrinsic_price",
    "physical_vs_intrinsic_bubble_pct",
]].describe(include="all")

In [ ]:
fig = go.Figure(go.Scatter(x=physical_full_history['date'],
    y=physical_full_history['physical_vs_intrinsic_bubble_pct'], mode='lines+markers',
    marker_size=4, name='Observed IME physical premium', line_color='#7B1FA2'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title='Full-history IME physical copper vs intrinsic LME-FX value',
    xaxis_title='Physical trade date', yaxis_title='Premium / discount (%)',
    height=600, template='plotly_white')
fig.show()

## 12. Corrected regression: physical price on intrinsic LME–FX value

This section supersedes the earlier experimental two-feature regression. The sole explanatory variable is intrinsic copper value: `LME cash USD/kg × free-market USD/IRR`. Certificate price is not used in model fitting. Three specifications are compared: proportional (no intercept), linear with intercept, and degree-2 polynomial Ridge.

In [ ]:
INTRINSIC_FEATURE = ["intrinsic_price"]
X_intrinsic = anchors[INTRINSIC_FEATURE]
y_physical = anchors["physical_price"]

intrinsic_models = {
    "proportional_no_intercept": LinearRegression(fit_intercept=False),
    "linear_with_intercept": LinearRegression(),
    "polynomial_degree_2_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-4, 4, 81))),
    ]),
}

intrinsic_comparison_rows = []
intrinsic_oos_predictions = {}
intrinsic_tscv = TimeSeriesSplit(n_splits=5)

for name, model in intrinsic_models.items():
    prediction = pd.Series(np.nan, index=y_physical.index, dtype=float)
    for train_index, test_index in intrinsic_tscv.split(X_intrinsic):
        fold_model = clone(model)
        fold_model.fit(X_intrinsic.iloc[train_index], y_physical.iloc[train_index])
        prediction.iloc[test_index] = fold_model.predict(X_intrinsic.iloc[test_index])
    valid = prediction.notna()
    actual = y_physical.loc[valid]
    predicted = prediction.loc[valid]
    intrinsic_oos_predictions[name] = predicted
    intrinsic_comparison_rows.append({
        "model": name,
        "oos_n": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    })

intrinsic_model_comparison = (
    pd.DataFrame(intrinsic_comparison_rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
intrinsic_model_comparison

In [ ]:
selected_intrinsic_model_name = intrinsic_model_comparison.loc[0, "model"]
selected_intrinsic_model = clone(intrinsic_models[selected_intrinsic_model_name])
selected_intrinsic_model.fit(X_intrinsic, y_physical)

anchors_intrinsic_regression = anchors.copy()
anchors_intrinsic_regression["fitted_physical_price"] = (
    selected_intrinsic_model.predict(anchors_intrinsic_regression[INTRINSIC_FEATURE])
)
anchors_intrinsic_regression["fitted_error_pct"] = (
    anchors_intrinsic_regression["fitted_physical_price"]
    / anchors_intrinsic_regression["physical_price"]
    - 1
) * 100

bubble_intrinsic_regression = certificate_features.copy()
bubble_intrinsic_regression["estimated_physical_price"] = (
    selected_intrinsic_model.predict(bubble_intrinsic_regression[INTRINSIC_FEATURE])
)
if (bubble_intrinsic_regression["estimated_physical_price"] <= 0).any():
    raise ValueError("The selected intrinsic regression produced a non-positive price")
bubble_intrinsic_regression["certificate_bubble_pct"] = (
    bubble_intrinsic_regression["certificate_price"]
    / bubble_intrinsic_regression["estimated_physical_price"]
    - 1
) * 100
bubble_intrinsic_regression["is_physical_anchor_date"] = (
    bubble_intrinsic_regression["date"].isin(anchors["date"])
)

print(f"Selected corrected model: {selected_intrinsic_model_name}")
anchors_intrinsic_regression[[
    "date", "intrinsic_price", "physical_price",
    "fitted_physical_price", "fitted_error_pct"
]]

In [ ]:
fit_limits = [min(anchors_intrinsic_regression['physical_price'].min(),
                  anchors_intrinsic_regression['fitted_physical_price'].min()),
              max(anchors_intrinsic_regression['physical_price'].max(),
                  anchors_intrinsic_regression['fitted_physical_price'].max())]
anchor_regression_bubbles = bubble_intrinsic_regression.loc[
    bubble_intrinsic_regression['is_physical_anchor_date']]
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'Observed vs fitted: {selected_intrinsic_model_name}', 'Certificate bubble: intrinsic regression'))
fig.add_trace(go.Scatter(x=anchors_intrinsic_regression['physical_price'],
    y=anchors_intrinsic_regression['fitted_physical_price'], mode='markers',
    name='Physical anchors', marker_color='#1976D2'), row=1, col=1)
fig.add_trace(go.Scatter(x=fit_limits, y=fit_limits, name='Perfect fit',
    line=dict(color='#455A64', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=bubble_intrinsic_regression['date'],
    y=bubble_intrinsic_regression['certificate_bubble_pct'],
    name='Regression bubble', line_color='#7B1FA2'), row=1, col=2)
fig.add_trace(go.Scatter(x=anchor_regression_bubbles['date'],
    y=anchor_regression_bubbles['certificate_bubble_pct'], mode='markers',
    name='Physical anchor date', marker_color='#C62828'), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='#455A64', row=1, col=2)
fig.update_xaxes(title_text='Observed physical price (IRR/kg)', row=1, col=1)
fig.update_yaxes(title_text='Fitted physical price (IRR/kg)', row=1, col=1)
fig.update_xaxes(title_text='Date', row=1, col=2)
fig.update_yaxes(title_text='Bubble (%)', row=1, col=2)
fig.update_layout(height=550, template='plotly_white')
fig.show()

## 13. Forward trades inside the 102-day cash gap

The gap is a gap in the exact NCI copper-cathode cash benchmark, not a market closure. This section isolates exact-symbol forward trades strictly inside the gap and compares their weighted prices with the last cash price before the gap, the first cash price after it, and a straight-line bridge between those two cash anchors.

In [ ]:
forward_gap_path = first_existing(
    "../data/processed/physical/nci_copper_forward_gap.csv",
    "commodity/copper/data/processed/physical/nci_copper_forward_gap.csv",
)
forward_gap = pd.read_csv(forward_gap_path)
forward_gap["date"] = pd.to_datetime(forward_gap["trade_date_gregorian"])

forward_gap_summary = pd.Series(
    {
        "forward trade dates": len(forward_gap),
        "total quantity (t)": forward_gap["total_quantity"].sum(),
        "quantity-weighted price (IRR/kg)": np.average(
            forward_gap["forward_weighted_price"],
            weights=forward_gap["total_quantity"],
        ),
        "previous cash price (IRR/kg)": forward_gap["previous_cash_price"].iloc[0],
        "next cash price (IRR/kg)": forward_gap["next_cash_price"].iloc[0],
    },
    name="value",
)
display(forward_gap_summary.to_frame())
display(
    forward_gap[
        [
            "trade_date_jalali",
            "contract_types",
            "total_quantity",
            "forward_weighted_price",
            "vs_previous_cash_pct",
            "vs_next_cash_pct",
            "vs_linear_bridge_pct",
        ]
    ].round(2)
)

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
    subplot_titles=('Forward prices inside the 102-day cash gap', 'Difference from cash anchors'))
dates = forward_gap['trade_date_jalali']
for column, name, color in [
    ('forward_weighted_price', 'Forward weighted price', '#1976D2'),
    ('linear_cash_bridge_price', 'Linear cash bridge', '#EF6C00'),
]:
    fig.add_trace(go.Scatter(x=dates, y=forward_gap[column], name=name,
        mode='lines+markers', line_color=color), row=1, col=1)
for column, name, color in [
    ('previous_cash_price', 'Previous cash anchor', '#546E7A'),
    ('next_cash_price', 'Next cash anchor', '#2E7D32'),
]:
    fig.add_hline(y=forward_gap[column].iloc[0], line_color=color,
        annotation_text=name, row=1, col=1)
for column, name, color in [
    ('vs_previous_cash_pct', 'vs previous cash', '#1976D2'),
    ('vs_next_cash_pct', 'vs next cash', '#00897B'),
    ('vs_linear_bridge_pct', 'vs linear bridge', '#EF6C00'),
]:
    fig.add_trace(go.Bar(x=dates, y=forward_gap[column], name=name,
        marker_color=color), row=2, col=1)
fig.add_hline(y=0, line_color='#455A64', row=2, col=1)
fig.update_yaxes(title_text='IRR/kg', row=1, col=1)
fig.update_yaxes(title_text='Difference (%)', row=2, col=1)
fig.update_xaxes(title_text='Trade date (Jalali)', row=2, col=1)
fig.update_layout(height=850, barmode='group', template='plotly_white')
fig.show()

**Interpretation.** The 16 forward-trade dates total 26,420 tonnes. Forward prices range from 7.16% to 33.73% above the previous cash anchor and from 14.05% below to 7.26% above the next cash anchor. These observations are forward contracts with cash/credit settlement terms; they should not be inserted into the primary cash-only benchmark without explicit maturity and financing adjustments.

## 14. Certificate trading volume through time

The table and chart use every positive-volume copper-cathode certificate trading day in the canonical certificate input. Volume is the number of traded warehouse receipts reported by IME.

In [ ]:
certificate_volume_history = (
    certificate[["date", "TradesVolume", "TradesValue"]]
    .rename(
        columns={
            "TradesVolume": "certificate_trades_volume",
            "TradesValue": "certificate_trades_value_irr",
        }
    )
    .sort_values("date")
    .reset_index(drop=True)
)
assert certificate_volume_history["certificate_trades_volume"].gt(0).all()
display(certificate_volume_history)

In [ ]:
fig = go.Figure(go.Bar(x=certificate_volume_history['date'],
    y=certificate_volume_history['certificate_trades_volume'], marker_color='#1976D2'))
fig.update_layout(title='Copper-cathode certificate trading volume',
    xaxis_title='Trade date', yaxis_title='Traded certificates',
    height=550, template='plotly_white')
fig.show()

## 15. Certificate bubble on jointly observed dates only

This diagnostic removes all interpolated dates. Bubble is recalculated only when certificate and comparable NCI cash physical prices are both observed on the same day: `(certificate price / observed physical price - 1) × 100`.

In [ ]:
observed_certificate_bubble = approved_bubble.loc[
    approved_bubble["physical_ratio_method"].eq("observed"),
    [
        "date",
        "certificate_trades_volume",
        "certificate_price_irr_per_kg",
        "observed_physical_price_irr_per_kg",
    ],
].copy()
observed_certificate_bubble["observed_certificate_bubble_irr_per_kg"] = (
    observed_certificate_bubble["certificate_price_irr_per_kg"]
    - observed_certificate_bubble["observed_physical_price_irr_per_kg"]
)
observed_certificate_bubble["observed_certificate_bubble_pct"] = (
    observed_certificate_bubble["certificate_price_irr_per_kg"]
    / observed_certificate_bubble["observed_physical_price_irr_per_kg"]
    - 1
) * 100
observed_certificate_bubble = observed_certificate_bubble.sort_values("date").reset_index(drop=True)
assert len(observed_certificate_bubble) == approved_bubble["physical_ratio_method"].eq("observed").sum()
display(observed_certificate_bubble.round(2))

In [ ]:
colors = np.where(observed_certificate_bubble['observed_certificate_bubble_pct'].ge(0),
    '#C62828', '#2E7D32')
fig = go.Figure(go.Bar(x=observed_certificate_bubble['date'],
    y=observed_certificate_bubble['observed_certificate_bubble_pct'],
    marker_color=colors, name='Observed exact-date bubble'))
fig.add_hline(y=0, line_dash='dash', line_color='#455A64')
fig.update_layout(title=f'Copper certificate bubble: {len(observed_certificate_bubble)} exact-date anchors',
    xaxis_title='Trade date', yaxis_title='Observed bubble (%)',
    height=550, template='plotly_white')
fig.show()

## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Copper, product comparability still follows the project-specific workflow.

In [ ]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "copper").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "copper"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "copper", physical_filename='copper_cathode_physical_raw.csv'
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Copper")
plot_market_prices(physical_dashboard, certificate_dashboard, "Copper")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Copper", top_n=30)
display(goods_count_table)
# Approved comparison charts are displayed at the start of this notebook.

## Historical bubble distribution

This section reads the standardized processed table and renders an interactive Plotly figure for
each bubble type. The panels show the observed distribution, empirical cumulative distribution
function F(x), and magnitude frequency P(|Bubble| >= |x|). Negative bubbles retain their sign in
the first two panels; the third panel measures magnitude only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def locate_distribution_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "shared").exists() and (candidate / "commodity/copper").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

distribution_workspace = locate_distribution_workspace()
if str(distribution_workspace) not in sys.path:
    sys.path.insert(0, str(distribution_workspace))

from shared.market_analysis.bubble_distribution import plot_distribution_plotly

distribution_project = distribution_workspace / "commodity/copper"
distribution_files = list(
    (distribution_project / "data/processed/bubble").glob("*_bubble_distribution.csv")
)
if len(distribution_files) != 1:
    raise ValueError(f"Expected one named bubble distribution CSV, found {distribution_files}")
bubble_distribution = pd.read_csv(distribution_files[0], parse_dates=["observation_date"])
for series_id, series_distribution in bubble_distribution.groupby("series_id", sort=True):
    comparison = series_distribution["comparison"].iloc[0]
    figure = plot_distribution_plotly(series_distribution, comparison)
    figure.show()

display(bubble_distribution)